HWモデル+BSモデルでの株価SDEを考える
```math
dS_t = \mu S_t dt + \sigma_{S} S_t dW^{S} \\
dB_t = r_t B_t dt \\
dr_t = \left( \theta_{r}(t) - \alpha_{r}(t) r_t \right) + \sigma_{r}(t) dW^{r} \\
dW^{S} dW^{r} = \rho^{r,S} dt
```
$B_t$を基準材とするQリスク中立測度でのSDEを考える
```math
dS_t = r_t S_t dt + \sigma_{S} S_t dW^{S,Q} \\
dB_t = r_t B_t dt \\
dr_t = \left( \theta_{r}(t) - \alpha_{r}(t) r_t \right) + \sigma_{r}(t) dW^{r,Q} \\
dW^{S} dW^{r} = \rho^{r,S} dt
```
それぞれ解析解を考える。
```math
DF(t,T) = \frac{B_t}{B_T} = \exp \left( - \int_{t}^{T} r_s ds \right)\\
S_T = S_t \exp \left( \int_{t}^{T} r_s ds - \frac{1}{2} \int_{t}^{T} \sigma_{S}^{2} ds + \sigma \int_{t}^{T} dW^{S,Q} \right) \\
B_T = B_t \exp \left( \int_{t}^{T} r_s ds \right)
```
$T$セトル$K$執行のフォワード取引のPV
```math
Fwd(t,T) = \text{E}^{Q} \left[ DF(t,T) (S_T - K) \right]
```
$T$セトル$K$ストライクのコールオプションのPV
```math
Call(t,K,T) = \text{E}^Q \left[ DF(t,T) \text{max}(S_T - K, 0) \right]
```
$T$セトルのゼロクーポン債のPV
```math
P(t,T) = \text{E}^Q [ DF(t,T) ]
```
ゼロクーポン債券の解析解
```math
P(t,T) = \exp \left( A(t,T) - B_{r}(t,T) r_t  \right) \\[10pt]
A(t,T) = \ln \frac{P(0,T)}{P(0,t)} + B_{r}(t,T) f(0,t) - \frac{1}{2} \int_{t}^{T} \sigma_{r}^{2}(u) B_{r}(u,T)^2 du \\
A(t,T) = -\int_{t}^{T} B_{r}(u,T) \theta_{r}(u) du + \frac{1}{2} \int_{t}^{T} \sigma_{r}^{2}(u) B_{r}(u,T)^2 du \\
B_{r}(t,T) = \int_{t}^{T} \exp \left( -\int_{t}^{u} \alpha_{r}(s) ds \right) du
```




ゼロクーポン債券自体のSDEは次のように書ける。
```math
\frac{dP(t,T)}{P(t,T)} = r_t  dt - \sigma_{r}(t) B_{r}(t, T) dW^{r,Q}
```



デフォルトを考慮したゼロクーポン債のPV (ただし$P_d (t,T)$は$t$までにデフォルトしなかった場合の条件付き価格・マルチンゲールであるために必要)
```math
P_d(t,T) \exp \left( -\int_{-\infty}^{t} \lambda_s ds \right) = \text{E}^Q \left[DF(t,T) \exp \left( -\int_{-\infty}^{T} \lambda_s ds \right) \right]
```
定式化は
```math
P_d(t,T) = \text{E}^Q \left[ \exp \left( -\int_{t}^{T} (r_t + \lambda_s) ds \right) \right]
```
ハザードレートSDEを考える。
```math
dr_t = \left( \theta_{r}(t) - \alpha_{r}(t) r_t \right) + \sigma_{r}(t) dW^{r,Q} \\
d\lambda_t = \left( \theta_{\lambda}(t) - \alpha_{\lambda}(t) \lambda_t \right) + \sigma_{\lambda}(t) dW^{\lambda,Q} \\
dW^{r,Q} dW^{\lambda,Q} = \rho^{r,\lambda} dt
```
デフォルト考慮ゼロクーポン債券の解析解
```math
P_d(t,T) = \exp \left( A_d(t,T) - B_{r}(t,T) r_t - B_{\lambda}(t,T) \lambda_t \right) \\[10pt]
A_{d} (t,T) = A_{r}(t,T) + A_{\lambda}(t,T) + A_{r,\lambda}(t,T) \\
A_{r} (t,T) = -\int_{t}^{T} B_{r}(u,T) \theta_{r}(u) du + \frac{1}{2} \int_{t}^{T} \sigma_{r}^{2}(u) B_{r}(u,T)^2 du \\
A_{\lambda} (t,T) = -\int_{t}^{T} B_{\lambda}(u,T) \theta_{\lambda}(u) du + \frac{1}{2} \int_{t}^{T} \sigma_{\lambda}^{2}(u) B_{\lambda}(u,T)^2 du \\
A_{r,\lambda} (t,T) = \frac{1}{2} \int_{t}^{T} 2 \rho_{r,\lambda} \sigma_{r} (u) \sigma_{\lambda} (u) B_{r}(u,T) B_{\lambda}(u,T) du \\
B_{r}(t,T) = \int_{t}^{T} \exp \left( -\int_{t}^{u} \alpha_{r}(s) ds \right) du \\
B_{\lambda}(t,T) = \int_{t}^{T} \exp \left( -\int_{t}^{u} \alpha_{\lambda}(s) ds \right) du
```

In [36]:
import numpy as np
from scipy.interpolate import CubicSpline

class HullWhiteTheta:
    def __init__(self, times, dfs, alpha, sigma):
        """
        times : array-like
            割引債の満期時刻 T_i
        dfs : array-like
            DF(0, T_i)
        alpha : float or callable
            mean reversion α(t)
        sigma : float or callable
            volatility σ(t)
        """

        self.times = np.array(times)
        self.log_dfs = np.log(dfs)

        # log DF の spline 補間
        self.log_df_spline = CubicSpline(self.times, self.log_dfs)

        # α(t), σ(t) を関数として統一
        self.alpha = alpha if callable(alpha) else lambda t: alpha
        self.sigma = sigma if callable(sigma) else lambda t: sigma

    def fwd_rate(self, t):
        """ f(0,t) """
        return -self.log_df_spline.derivative(1)(t)

    def dfwd_dt(self, t):
        """ ∂f(0,t)/∂t """
        return -self.log_df_spline.derivative(2)(t)

    def theta(self, t):
        """
        Hull–White の θ(t)
        """
        alpha_t = self.alpha(t)
        sigma_t = self.sigma(t)

        f0t = self.fwd_rate(t)
        dfdt = self.dfwd_dt(t)

        # 分散補正項
        var_term = (
            sigma_t**2 / alpha_t**2
            * (1.0 - np.exp(-alpha_t * t))**2
        )

        # 時間微分（解析的に書いても良いが、ここでは数値安定性優先）
        dvar_dt = (
            2 * sigma_t**2 / alpha_t
            * (1.0 - np.exp(-alpha_t * t))
            * np.exp(-alpha_t * t)
        )

        return dfdt + alpha_t * f0t + 0.5 * dvar_dt
    
    def df(self, t):
        """ DF(0,t) """
        return np.exp(self.log_df_spline(t))
    
    
if __name__ == "__main__":
    # Example usage
    times = [0.5, 1.0, 2.0, 5.0, 10.0]
    dfs = [0.99, 0.975, 0.94, 0.85, 0.70]
    alpha = 0.1
    sigma = 0.0

    hw_theta = HullWhiteTheta(times, dfs, alpha, sigma)

    t = 2.5
    print("Theta at t =", t, ":", hw_theta.theta(t))
    print("DF at t =", t, ":", hw_theta.df(t))
    print("Fwd rate at t =", t, ":", hw_theta.fwd_rate(t))

Theta at t = 2.5 : 0.0005155329123274293
DF at t = 2.5 : 0.9231474035798196
Fwd rate at t = 2.5 : 0.03536941888215867


In [ ]:
import numpy as np

class HWBSMonteCarlo:
    def __init__(
        self,
        S0,
        r0,
        lambda0,
        sigma_s,
        alpha_r,
        sigma_r,
        theta_r_func,
        alpha_l,
        sigma_l,
        theta_l_func,
        rho_sr,
        rho_sl,
        rho_rl,
        T,
        n_steps,
        n_paths,
        seed=42,
    ):
        self.S0 = S0
        self.r0 = r0
        self.lambda0 = lambda0

        self.sigma_s = sigma_s

        self.alpha_r = alpha_r
        self.sigma_r = sigma_r
        self.theta_r_func = theta_r_func

        self.alpha_l = alpha_l
        self.sigma_l = sigma_l
        self.theta_l_func = theta_l_func

        self.T = T
        self.n_steps = n_steps
        self.n_paths = n_paths
        self.dt = T / n_steps
        self.time_grid = np.linspace(0, T, n_steps + 1)

        np.random.seed(seed)

        # Correlation matrix (S, r, lambda)
        corr = np.array([
            [1.0,     rho_sr, rho_sl],
            [rho_sr,  1.0,    rho_rl],
            [rho_sl,  rho_rl, 1.0]
        ])

        self.chol = np.linalg.cholesky(corr)


    # -----------------------------
    # Hull-White helpers
    # -----------------------------
    def B(self, t, T, alpha):
        return (1.0 - np.exp(-alpha * (T - t))) / alpha
    
    def B_r(self, t, T):
        return self.B(t, T, self.alpha_r)
    
    def B_l(self, t, T):
        return self.B(t, T, self.alpha_l)


    def A(self, t, T, alpha, sigma, theta_func, n_int=50):
        if t >= T:
            return 0.0

        u = np.linspace(t, T, n_int)
        du = (T - t) / (n_int - 1)

        B_uT = self.B(u, T, alpha)
        theta_u = np.array([theta_func(x) for x in u])

        term1 = -np.sum(B_uT * theta_u) * du
        term2 = 0.5 * np.sum((sigma * B_uT) ** 2) * du
        return term1 + term2
    
    def A_r(self, t, T, n_int=50):
        return self.A(t, T, self.alpha_r, self.sigma_r, self.theta_r_func, n_int)
    
    def A_l(self, t, T, n_int=50):
        return self.A(t, T, self.alpha_l, self.sigma_l, self.theta_l_func, n_int)
    
    def A_d(self, t, T, n_int=50):
        if t >= T:
            return 0.0
        
        u = np.linspace(t, T, n_int)
        du = (T - t) / (n_int - 1)

        Br = self.B_r(u, T)
        Bl = self.B_l(u, T)

        Arl = np.sum(
            self.sigma_r * self.sigma_l * Br * Bl
        ) * self.rho_rl * du

        return Arl



    def simulate_paths(self):
        S = np.zeros((self.n_paths, self.n_steps + 1))
        r = np.zeros((self.n_paths, self.n_steps + 1))
        l = np.zeros((self.n_paths, self.n_steps + 1))
        df_r = np.ones((self.n_paths, self.n_steps + 1))
        df_l = np.ones((self.n_paths, self.n_steps + 1))

        S[:, 0] = self.S0
        r[:, 0] = self.r0
        l[:, 0] = self.lambda0

        for i in range(self.n_steps):
            t = self.time_grid[i]

            Z = np.random.randn(self.n_paths, 3)
            dW = Z @ self.chol.T * np.sqrt(self.dt)

            dW_s = dW[:, 0]
            dW_r = dW[:, 1]
            dW_l = dW[:, 2]

            # short rate
            r[:, i + 1] = (
                r[:, i]
                + (self.theta_r_func(t) - self.alpha_r * r[:, i]) * self.dt
                + self.sigma_r * dW_r
            )

            # hazard rate
            l[:, i + 1] = (
                l[:, i]
                + (self.theta_l_func(t) - self.alpha_l * l[:, i]) * self.dt
                + self.sigma_l * dW_l
            )

            # stock
            S[:, i + 1] = S[:, i] * np.exp(
                (r[:, i] - 0.5 * self.sigma_s ** 2) * self.dt
                + self.sigma_s * dW_s
            )

            # discount factor
            df_r[:, i + 1] = df_r[:, i] * np.exp(-r[:, i] * self.dt)
            df_l[:, i + 1] = df_l[:, i] * np.exp(-l[:, i] * self.dt)

        return S, r, df_r, l, df_l



    def simulate_ZCB_paths(self, r, df, T_bond):
    
        P = np.zeros((self.n_paths, self.n_steps + 1))  # ZCB

        # initial ZCB
        A0 = self.A_r(0.0, T_bond)
        B0 = self.B_r(0.0, T_bond)
        P[:, 0] = np.exp(A0 - B0 * r[:, 0])

        for i in range(self.n_steps):
            t = self.time_grid[i]
            coeff = 1.0 if t <= T_bond else 0.0

            # zero-coupon bond
            A_t = self.A_r(t + self.dt, T_bond)
            B_t = self.B_r(t + self.dt, T_bond)
            P[:, i + 1] = np.exp(A_t - B_t * r[:, i + 1]) * coeff
        
        return P

    def price_forward(self, K):
        S, r, df_r, l, df_l = self.simulate_paths()
        payoff = S[:, -1] - K
        price = np.mean(df_r[:, -1] * payoff)
        return price

    def price_call(self, K):
        S, r, df_r, l, df_l = self.simulate_paths()
        payoff = np.maximum(S[:, -1] - K, 0.0)
        price = np.mean(df_r[:, -1] * payoff)
        return price
    
    def price_ZCB(self, T_bond):
        _, r, df_r, l, df_l = self.simulate_paths()
        P = self.simulate_ZCB_paths(r, df_r, T_bond)
        payoff = 1.0
        price = np.mean(P[:, 0] * payoff)
        return price
    
    
    def getT(self):
        return self.time_grid
    
    def getDF(self):
        _, _, df_r, _, df_l = self.simulate_paths()
        return np.mean(df_r, axis=0)

    def getS(self):
        S, _, _, _, _ = self.simulate_paths()
        return np.mean(S, axis=0)
    
    def getB(self):
        _, _, df_r, _, df_l = self.simulate_paths()
        B = 1.0 / df_r
        return np.mean(B, axis=0)
    
    def getr(self):
        _, r, _, _, _ = self.simulate_paths()
        return np.mean(r, axis=0)
    
    def getZCB(self, T_bond):
        _, r, df_r, _, df_l = self.simulate_paths()
        P = self.simulate_ZCB_paths(r, df_r, T_bond)
        return np.mean(P, axis=0)
    
    def getDF_paths(self):
        _, _, df_r, _, df_l = self.simulate_paths()
        return df_r

    def getS_paths(self):
        S, _, _, _, _ = self.simulate_paths()
        return S
    
    def getB_paths(self):
        _, _, df_r, _, df_l = self.simulate_paths()
        B = 1.0 / df_r
        return B
    
    def getr_paths(self):
        _, r, _, _, _ = self.simulate_paths()
        return r
    
    def getZCB_paths(self, T_bond):
        _, r, df_r, _, df_l = self.simulate_paths()
        P = self.simulate_ZCB_paths(r, df_r, T_bond)
        return P
    

In [38]:
times = [0, 0.5, 1.0, 2.0, 5.0, 10.0]
dfs   = [1.00,0.99, 0.975, 0.94, 0.85, 0.70]

ALPHA_r = 0.03
SIGMA_r = 0.01

hw_r = HullWhiteTheta(times, dfs, ALPHA_r, SIGMA_r)

# theta(t) は任意
def theta_r(t):
    return hw_r.theta(t)



times = [0, 0.5, 1.0, 2.0, 5.0, 10.0]
dfs   = [1.00,0.99, 0.975, 0.94, 0.85, 0.70]

ALPHA_l = 0.03
SIGMA_l = 0.01

hw_l = HullWhiteTheta(times, dfs, ALPHA_l, SIGMA_l)

# theta(t) は任意
def theta_l(t):
    return hw_l.theta(t)


engine = HWBSMonteCarlo(
    S0=100.0,
    r0= hw_r.fwd_rate(0.0),
    lambda0=hw_l.fwd_rate(0.0),
    sigma_s=0.2,
    alpha_r=ALPHA_r,
    sigma_r=SIGMA_r,
    theta_r_func=theta_r,
    alpha_l=ALPHA_l,
    sigma_l=SIGMA_l,
    theta_l_func=theta_l,
    rho_sr=0.3,
    rho_sl=0.2,
    rho_rl=0.1,
    T=10.0,
    n_steps=500,
    n_paths=10000,
)

K = 100.0


#print("Forward price:", engine.price_forward(K))
#print("Call price:", engine.price_call(K))
#print("ZCB price:", engine.price_ZCB(5))
print("Discount Factor Curve:", engine.getDF())
#print("S Curve:", engine.getS())
#print("r Curve:", engine.getr())
#print("Time Grid:", engine.getT())
#print("S Paths:", engine.getS_paths())
#print("r Paths:", engine.getr_paths())
#print("DF Paths:", engine.getDF_paths())

Discount Factor Curve: [1.         0.99973933 0.99946592 0.99918013 0.99888191 0.99857123
 0.99824889 0.99791451 0.99756862 0.99721128 0.9968432  0.99646416
 0.99607463 0.99567466 0.99526375 0.99484318 0.99441267 0.99397247
 0.99352312 0.99306426 0.99259585 0.99211856 0.99163202 0.99113652
 0.99063223 0.99011959 0.98959883 0.98907028 0.9885339  0.98799015
 0.98743895 0.98688052 0.9863151  0.98574345 0.98516543 0.98458074
 0.98398998 0.98339284 0.98278992 0.98218128 0.98156672 0.98094671
 0.98032168 0.9796923  0.97905822 0.9784203  0.97777788 0.97713163
 0.97648084 0.97582619 0.97516804 0.97450645 0.97384155 0.97317333
 0.97250174 0.97182721 0.97115005 0.97046968 0.96978673 0.96910157
 0.96841386 0.96772386 0.96703177 0.96633712 0.96564057 0.9649421
 0.96424145 0.96353881 0.96283425 0.96212818 0.96142099 0.96071243
 0.96000233 0.95929122 0.95857872 0.95786543 0.95715136 0.95643629
 0.95572038 0.95500413 0.95428744 0.95357054 0.95285321 0.95213543
 0.95141837 0.95070158 0.94998533 0.9492

In [39]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def plot_df_curve(alpha_r, sigma_r):
    # Hull-White theta 再構築
    hw_r = HullWhiteTheta(times, dfs, alpha_r, sigma_r)
    
    def theta_r(t):
        return hw_r.theta(t)
    

    ALPHA_l = 0.03
    SIGMA_l = 0.01

    hw_l = HullWhiteTheta(times, dfs, ALPHA_l, SIGMA_l)

    # theta(t) は任意
    def theta_l(t):
        return hw_l.theta(t)
    

    engine = HWBSMonteCarlo(
        S0=100.0,
        r0= hw_r.fwd_rate(0.0),
        lambda0=hw_l.fwd_rate(0.0),
        sigma_s=0.2,
        alpha_r=alpha_r,
        sigma_r=sigma_r,
        theta_r_func=theta_r,
        alpha_l=ALPHA_l,
        sigma_l=SIGMA_l,
        theta_l_func=theta_l,
        rho_sr=0.3,
        rho_sl=0.2,
        rho_rl=0.1,
        T=10.0,
        n_steps=500,
        n_paths=10000)

    time_grid = engine.getT()
    df_curve = engine.getDF()
    df_curve_theoretical = hw_r.df(time_grid)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=df_curve,
            mode="lines",
            name="Discount Factor",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=df_curve_theoretical,
            mode="lines",
            name="Theoretical Discount Factor",
        )
    )

    fig.update_layout(
        title=f"HW Discount Factor Curve (alpha={alpha_r:.3f}, sigma={sigma_r:.3f})",
        xaxis_title="Time (T)",
        yaxis_title="Discount Factor",
        yaxis=dict(range=[0, 1.05]),
        template="plotly_white",
    )

    fig.show()

alpha_r_slider = widgets.FloatSlider(
    value=ALPHA_r,
    min=0.001,
    max=0.2,
    step=0.005,
    description="alpha_r",
    continuous_update=False,
)

sigma_r_slider = widgets.FloatSlider(
    value=SIGMA_r,
    min=0.001,
    max=0.05,
    step=0.002,
    description="sigma_r",
    continuous_update=False,
)

ui = widgets.VBox([alpha_r_slider, sigma_r_slider])

out = widgets.interactive_output(
    plot_df_curve,
    {
        "alpha_r": alpha_r_slider,
        "sigma_r": sigma_r_slider,
    },
)

display(ui, out)



Output()

In [40]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def plot_asset_curve(alpha_r, sigma_r, sigma_s, rho, T_bond):
    # Hull-White theta 再構築
    hw_r = HullWhiteTheta(times, dfs, alpha_r, sigma_r)
    
    def theta_r(t):
        return hw_r.theta(t)
    

    ALPHA_l = 0.03
    SIGMA_l = 0.01

    hw_l = HullWhiteTheta(times, dfs, ALPHA_l, SIGMA_l)

    # theta(t) は任意
    def theta_l(t):
        return hw_l.theta(t)



    engine = HWBSMonteCarlo(
        S0=1.0,
        r0= hw_r.fwd_rate(0.0),
        lambda0=hw_l.fwd_rate(0.0),
        sigma_s=0.2,
        alpha_r=alpha_r,
        sigma_r=sigma_r,
        theta_r_func=theta_r,
        alpha_l=ALPHA_l,
        sigma_l=SIGMA_l,
        theta_l_func=theta_l,
        rho_sr=rho,
        rho_sl=0.2,
        rho_rl=0.1,
        T=10.0,
        n_steps=500,
        n_paths=10000)

    time_grid = engine.getT()
    B_t = engine.getB()
    S_t = engine.getS()
    ZCB_t = engine.getZCB(T_bond)


    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=S_t,
            mode="lines",
            name="Stock Price",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=B_t,
            mode="lines",
            name="Money Market Account",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=ZCB_t,
            mode="lines",
            name=f"ZCB(t,{T_bond}) Price",
        )
    )

    fig.update_layout(
        title=f"Asset Price Evolution (alpha={alpha_r:.3f}, sigma={sigma_r:.3f}, sigma_s={sigma_s:.3f}, rho={rho:.2f}, T_bond={T_bond})",
        xaxis_title="Time (T)",
        yaxis_title="Asset Price",
        template="plotly_white",
    )

    fig.show()

alpha_r_slider = widgets.FloatSlider(
    value=ALPHA_r,
    min=0.001,
    max=0.2,
    step=0.005,
    description="alpha_r",
    continuous_update=False,
)

sigma_r_slider = widgets.FloatSlider(
    value=SIGMA_r,
    min=0.001,
    max=0.05,
    step=0.002,
    description="sigma_r",
    continuous_update=False,
)

sigma_s_slider = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=0.5,
    step=0.01,
    description="sigma_s",
    continuous_update=False,
)

rho_slider = widgets.FloatSlider(
    value=0.3,
    min=-1.0,
    max=1.0,
    step=0.05,
    description="rho",
    continuous_update=False,
)

T_bond_slider = widgets.FloatSlider(
    value=5,
    min=1.0,
    max=10.0,
    step=0.05,
    description="T_bond",
    continuous_update=False,
)

ui = widgets.VBox([alpha_r_slider, sigma_r_slider, sigma_s_slider, rho_slider, T_bond_slider])

out = widgets.interactive_output(
    plot_asset_curve,
    {
        "alpha_r": alpha_r_slider,
        "sigma_r": sigma_r_slider,
        "sigma_s": sigma_s_slider,
        "rho": rho_slider,
        "T_bond": T_bond_slider,
    },
)

display(ui, out)



Output()

In [41]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def plot_call_price_vs_S(
    T,
    alpha_r,
    sigma_r,
    sigma_s,
    K,
    rho,
):
    # Hull-White theta 再構築
    hw_r = HullWhiteTheta(times, dfs, alpha_r, sigma_r)
    
    def theta_r(t):
        return hw_r.theta(t)
    

    ALPHA_l = 0.03
    SIGMA_l = 0.01

    hw_l = HullWhiteTheta(times, dfs, ALPHA_l, SIGMA_l)

    # theta(t) は任意
    def theta_l(t):
        return hw_l.theta(t)


    # S0 grid（横軸）
    S0_grid = np.linspace(0.5, 1.5, 25)
    prices = []

    for S0 in S0_grid:
        engine = HWBSMonteCarlo(
            S0=S0,
            r0= hw_r.fwd_rate(0.0),
            lambda0=hw_l.fwd_rate(0.0),
            sigma_s=sigma_s,
            alpha_r=alpha_r,
            sigma_r=sigma_r,
            theta_r_func=theta_r,
            alpha_l=ALPHA_l,
            sigma_l=SIGMA_l,
            theta_l_func=theta_l,
            rho_sr=rho,
            rho_sl=0.2,
            rho_rl=0.1,
            T=T,
            n_steps=500,
            n_paths=10000)



        price = engine.price_call(K)
        prices.append(price)

        

    prices = np.array(prices)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=S0_grid,
            y=prices,
            mode="lines+markers",
            name="Call Price",
        )
    )

    fig.update_layout(
        title=(
            f"Call Price vs S₀  "
            f"(T={T:.2f}, K={K:.2f}, "
            f"α={alpha_r:.3f}, σ_r={sigma_r:.3f}, "
            f"σ_s={sigma_s:.3f}, ρ={rho:.2f})"
        ),
        xaxis_title="Initial Stock Price S₀",
        yaxis_title="Call Price",
        template="plotly_white",
    )

    fig.show()


# =====================
# sliders
# =====================
T_slider = widgets.FloatSlider(
    value=5.0,
    min=0.5,
    max=10.0,
    step=0.5,
    description="T",
    continuous_update=False,
)

alpha_r_slider = widgets.FloatSlider(
    value=ALPHA_r,
    min=0.001,
    max=0.2,
    step=0.005,
    description="alpha_r",
    continuous_update=False,
)

sigma_r_slider = widgets.FloatSlider(
    value=SIGMA_r,
    min=0.001,
    max=0.05,
    step=0.002,
    description="sigma_r",
    continuous_update=False,
)

sigma_s_slider = widgets.FloatSlider(
    value=0.2,
    min=0.05,
    max=0.6,
    step=0.01,
    description="sigma_s",
    continuous_update=False,
)

K_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=1.5,
    step=0.02,
    description="K",
    continuous_update=False,
)

rho_slider = widgets.FloatSlider(
    value=0.0,
    min=-0.9,
    max=0.9,
    step=0.05,
    description="rho",
    continuous_update=False,
)

ui = widgets.VBox(
    [
        T_slider,
        alpha_r_slider,
        sigma_r_slider,
        sigma_s_slider,
        K_slider,
        rho_slider,
    ]
)

out = widgets.interactive_output(
    plot_call_price_vs_S,
    {
        "T": T_slider,
        "alpha_r": alpha_r_slider,
        "sigma_r": sigma_r_slider,
        "sigma_s": sigma_s_slider,
        "K": K_slider,
        "rho": rho_slider,
    },
)

display(ui, out)


Output()